# 01 — Préparation des Données

Pipeline de chargement, nettoyage, encodage et découpage du jeu de données accidents 2024.

## 1. Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 2. Chargement et Nettoyage

In [ ]:
df = pd.read_csv('../data/raw/accidents_2024.csv', sep=';', encoding='latin-1')

df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravité (label)'], inplace=True)

df['Gravité (label)'] = df['Gravité (label)'].replace({'O': '0_Indemne', '0': '0_Indemne'})

for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 3. Visualisation de la Variable Cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.countplot(
    data=df, x='Gravité (label)', ax=axes[0],
    order=df['Gravité (label)'].value_counts().index
)
axes[0].set_title('Répartition des classes de gravité (déséquilibre observé)')
axes[0].tick_params(axis='x', rotation=15)

df_graves = df[df['Gravité (label)'].isin(['Tué', 'Blessé hospitalisé'])]
sns.countplot(data=df_graves, x='Agglomération', hue='Gravité (label)', ax=axes[1])
axes[1].set_title('Accidents graves selon la zone (agglomération)')

plt.tight_layout()
plt.show()

## 4. Encodage One-Hot, Découpage Train/Test et Sauvegarde

In [ ]:
y_raw = df['Gravité (label)']
X_raw = df.drop(columns=['Num_Acc', 'Gravité (label)', 'Département'])

colonnes_categorielles = X_raw.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X_raw, columns=colonnes_categorielles, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_raw
)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print(f"Train : {X_train.shape}, Test : {X_test.shape}")
print("Fichiers sauvegardés dans data/processed/")